<a href="https://colab.research.google.com/github/simnawani1997-beep/paytm_capstone/blob/main/payments_fraud_analytics/reconcile_payments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

CATEGORIES = ["grocery", "food_delivery", "recharge", "bill_payment", "travel",
              "ecommerce", "entertainment"]
REGIONS = ["North", "South", "East", "West"]
METHODS = ["UPI", "Wallet", "Card", "Netbanking"]
METHOD_WEIGHTS = [0.55, 0.20, 0.15, 0.10]
AMOUNTS_INR = [49, 99, 149, 299, 499, 799, 1499, 2999, 4999]
AMOUNT_WEIGHTS = [0.18, 0.16, 0.14, 0.14, 0.12, 0.10, 0.08, 0.05, 0.03]

# --- 40 merchants ---
merchants = pd.DataFrame({
    "merchant_id": range(1, 41),
    "merchant_name": [f"Merchant_{i:03d}" for i in range(1, 41)],
    "category": [random.choice(CATEGORIES) for _ in range(40)],
    "region": [random.choice(REGIONS) for _ in range(40)],
})

# --- 350 established users, signed up 30-730 days before the window start ---
window_start = datetime(2026, 1, 1)
users = pd.DataFrame({
    "user_id": range(1, 351),
    "signup_date": [window_start - timedelta(days=random.randint(30, 730)) for _ in range(350)],
})

# --- 500 baseline transactions over a 30-day window ---
rows = []
for i in range(500):
    txn_time = window_start + timedelta(
        days=random.randint(0, 29), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    status = np.random.choice(["captured", "failed", "chargeback"], p=[0.92, 0.06, 0.02])
    rows.append({
        "transaction_id": f"TXN{100000+i}",
        "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": np.random.choice(AMOUNTS_INR, p=AMOUNT_WEIGHTS),
        "payment_method": np.random.choice(METHODS, p=METHOD_WEIGHTS),
        "status": status,
        "risk_score": random.randint(0, 100),
    })

# --- inject 15 "burner account" chargeback frauds: brand-new users (< 30 days old) ---
next_user_id = 351
for i in range(15):
    txn_time = window_start + timedelta(days=random.randint(10, 29), hours=random.randint(0, 23))
    signup = txn_time - timedelta(days=random.randint(1, 25))
    users = pd.concat([users, pd.DataFrame([{"user_id": next_user_id, "signup_date": signup}])],
                       ignore_index=True)
    rows.append({
        "transaction_id": f"TXN{200000+i}",
        "user_id": next_user_id,
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": random.choice([999, 1999, 2999, 4999]),
        "payment_method": "Card",
        "status": "chargeback",
        "risk_score": random.randint(70, 100),
    })
    next_user_id += 1

# --- inject 8 velocity-attack clusters: 4 rapid-fire txns each within a 5-minute window ---
for cluster in range(8):
    victim_user = random.randint(1, 350)
    base_time = window_start + timedelta(days=random.randint(0, 29), hours=random.randint(0, 23))
    for k in range(4):
        rows.append({
            "transaction_id": f"TXN{300000 + cluster*4 + k}",
            "user_id": victim_user,
            "merchant_id": random.randint(1, 40),
            "transaction_time": base_time + timedelta(minutes=k),
            "amount_inr": random.choice([299, 399, 499]),
            "payment_method": "Card",
            "status": "captured" if k == 3 else "failed",
            "risk_score": random.randint(60, 95),
        })

ledger = pd.DataFrame(rows)  # 500 + 15 + 32 = 547 rows
merchants.to_csv("merchants.csv", index=False)
users.to_csv("users.csv", index=False)
ledger.to_csv("ledger.csv", index=False)

# --- build the deliberately-discrepant "gateway export" copy for reconciliation ---
gateway = ledger.copy()
n = len(gateway)
missing_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
gateway = gateway.drop(index=missing_idx).reset_index(drop=True)

mismatch_idx = np.random.choice(len(gateway), size=int(0.03 * n), replace=False)
gateway.loc[mismatch_idx, "amount_inr"] = gateway.loc[mismatch_idx, "amount_inr"] + \
    np.random.choice([-100, -50, 50, 100], size=len(mismatch_idx))

extra_rows = []
for i in range(int(0.02 * n)):
    extra_rows.append({
        "transaction_id": f"TXNX{9000+i}", "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": window_start + timedelta(days=random.randint(0, 29)),
        "amount_inr": random.choice(AMOUNTS_INR), "payment_method": random.choice(METHODS),
        "status": "captured", "risk_score": random.randint(0, 100),
    })
gateway = pd.concat([gateway, pd.DataFrame(extra_rows)], ignore_index=True)

status_idx = np.random.choice(len(gateway), size=int(0.02 * n), replace=False)
gateway.loc[status_idx, "status"] = "failed"

gateway.to_csv("gateway_export.csv", index=False)

print("Data generation complete! 4 CSV files generated successfully.")

Data generation complete! 4 CSV files generated successfully.


In [3]:
import pandas as pd

def reconcile_payments(ledger_df, gateway_df):
    # Step 1: Find completely missing records using transaction_id sets
    ledger_ids = set(ledger_df['transaction_id'])
    gateway_ids = set(gateway_df['transaction_id'])

    missing_in_gateway_ids = ledger_ids - gateway_ids
    missing_in_ledger_ids = gateway_ids - ledger_ids

    # Filter the original tables to get rows for these missing IDs
    missing_in_gateway = ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_ids)]
    missing_in_ledger = gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger_ids)]

    # Step 2: Find rows present in BOTH systems to check for discrepancies
    common_ids = ledger_ids.intersection(gateway_ids)

    # Merge the rows matching these common IDs side-by-side
    common_ledger = ledger_df[ledger_df['transaction_id'].isin(common_ids)]
    common_gateway = gateway_df[gateway_df['transaction_id'].isin(common_ids)]

    merged = pd.merge(
        common_ledger[['transaction_id', 'amount_inr', 'status']],
        common_gateway[['transaction_id', 'amount_inr', 'status']],
        on='transaction_id',
        suffixes=('_ledger', '_gateway')
    )

    # Step 3: Identify Amount Mismatches & compute the exact difference
    amount_mismatches = merged[merged['amount_inr_ledger'] != merged['amount_inr_gateway']].copy()
    amount_mismatches['difference_amount'] = amount_mismatches['amount_inr_ledger'] - amount_mismatches['amount_inr_gateway']

    # Step 4: Identify Status Mismatches
    status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']]

    return missing_in_gateway, missing_in_ledger, amount_mismatches, status_mismatches

# --- Let's run it against your actual files ---
# Load your generated data files
df_ledger = pd.read_csv("ledger.csv")
df_gateway = pd.read_csv("gateway_export.csv")

# Run the auditor function
missing_gt, missing_ld, amt_mis, stat_mis = reconcile_payments(df_ledger, df_gateway)

# Print out a clean report summarizing the discrepancy counts for the graders
print("=== PAYTM PAYMENT RECONCILIATION DISCREPANCY REPORT ===")
print(f"1. Transactions missing in Gateway Export: {len(missing_gt)} rows")
print(f"2. Transactions missing in Internal Ledger: {len(missing_ld)} rows")
print(f"3. Value/Amount mismatches discovered:     {len(amt_mis)} rows")
print(f"4. Processing status mismatches discovered: {len(stat_mis)} rows")


=== PAYTM PAYMENT RECONCILIATION DISCREPANCY REPORT ===
1. Transactions missing in Gateway Export: 27 rows
2. Transactions missing in Internal Ledger: 10 rows
3. Value/Amount mismatches discovered:     16 rows
4. Processing status mismatches discovered: 9 rows
